# 19A FastFlow All-in-One

This notebook is designed for Modal or another external GPU environment. It is self-contained: it reads the raw `LSWMD.pkl`, prepares the shared anomaly-detection split, trains one or more FastFlow variants, and saves a comparison table in one run.

Default runtime note:

- batch size defaults to `32` as a safer starting point for an `A10`
- if your Modal run has comfortable VRAM headroom, the first thing to try is increasing batch size to `64`

Recommended single-sitting ablations:

- `wrn50_l23_s6`: main run, `layer2 + layer3`, `6` flow steps
- `wrn50_l2_s6`: tests whether the shallower feature map helps local defects
- `wrn50_l23_s4`: cheaper/lighter flow to see whether the main run is over-parameterized

Those three are a good cost-aware bundle. Score-reduction sweeps are already included and are much cheaper than retraining extra models.


In [1]:
from __future__ import annotations

import copy
import json
import math
import pickle
import random
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas.core.indexes as core_indexes
import torch
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from torchvision.models import Wide_ResNet50_2_Weights, wide_resnet50_2
from torchvision.models.feature_extraction import create_feature_extractor

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

CONFIG = {
    "run": {
        "raw_pickle": "LSWMD.pkl",
        "output_dir": "/output/fastflow_all_in_one",
        "seed": 42,
    },
    "split": {
        "image_size": 64,
        "train_normal_count": 50_000,
        "val_normal_count": 5_000,
        "test_normal_count": 5_000,
        "test_defect_fraction_of_test_normals": 0.05,
    },
    "training": {
        "device": "auto",
        "batch_size": 128,
        "num_workers": 0,
        "epochs": 16,
        "learning_rate": 5e-5,
        "weight_decay": 1e-5,
        "early_stopping_patience": 4,
        "grad_clip_norm": 0.5,
        "amp": False,
    },
    "scoring": {
        "threshold_quantile": 0.95,
        "sweep_variants": [
            {"name": "mean", "reduction": "mean", "topk_ratio": 0.10},
            {"name": "max", "reduction": "max", "topk_ratio": 0.10},
            {"name": "topk_r005", "reduction": "topk_mean", "topk_ratio": 0.05},
            {"name": "topk_r010", "reduction": "topk_mean", "topk_ratio": 0.10},
            {"name": "topk_r015", "reduction": "topk_mean", "topk_ratio": 0.15},
        ],
    },
    "variants": [
        {"name": "wrn50_l23_s6", "layers": ["layer2", "layer3"], "input_size": 224, "flow_steps": 6, "hidden_channels": 256, "scale_clamp": 1.3},
        {"name": "wrn50_l2_s6", "layers": ["layer2"], "input_size": 224, "flow_steps": 6, "hidden_channels": 256, "scale_clamp": 1.3},
        {"name": "wrn50_l23_s4", "layers": ["layer2", "layer3"], "input_size": 224, "flow_steps": 4, "hidden_channels": 256, "scale_clamp": 1.3},
    ],
}

CONFIG


{'run': {'raw_pickle': 'LSWMD.pkl',
  'output_dir': '/output/fastflow_all_in_one',
  'seed': 42},
 'split': {'image_size': 64,
  'train_normal_count': 50000,
  'val_normal_count': 5000,
  'test_normal_count': 5000,
  'test_defect_fraction_of_test_normals': 0.05},
 'training': {'device': 'auto',
  'batch_size': 128,
  'num_workers': 0,
  'epochs': 16,
  'learning_rate': 5e-05,
  'weight_decay': 1e-05,
  'early_stopping_patience': 4,
  'grad_clip_norm': 0.5,
  'amp': False},
 'scoring': {'threshold_quantile': 0.95,
  'sweep_variants': [{'name': 'mean', 'reduction': 'mean', 'topk_ratio': 0.1},
   {'name': 'max', 'reduction': 'max', 'topk_ratio': 0.1},
   {'name': 'topk_r005', 'reduction': 'topk_mean', 'topk_ratio': 0.05},
   {'name': 'topk_r010', 'reduction': 'topk_mean', 'topk_ratio': 0.1},
   {'name': 'topk_r015', 'reduction': 'topk_mean', 'topk_ratio': 0.15}]},
 'variants': [{'name': 'wrn50_l23_s6',
   'layers': ['layer2', 'layer3'],
   'input_size': 224,
   'flow_steps': 6,
   'hidden

In [2]:
LABEL_NORMAL = "none"
LABEL_DEFECT = "pattern"
LOG_2PI = math.log(2.0 * math.pi)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: str) -> torch.device:
    if device_name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_name)


def read_legacy_pickle(path: str | Path) -> pd.DataFrame:
    sys.modules["pandas.indexes"] = core_indexes
    with Path(path).open("rb") as handle:
        return pickle.load(handle, encoding="latin1")


def unwrap_legacy_value(value: Any) -> str:
    if value is None:
        return ""
    if hasattr(value, "size") and getattr(value, "size") == 0:
        return ""
    if hasattr(value, "tolist"):
        value = value.tolist()
    while isinstance(value, list) and len(value) == 1:
        value = value[0]
    return str(value).strip()


def normalize_map(wafer_map: np.ndarray, image_size: int) -> np.ndarray:
    wafer_map = np.asarray(wafer_map, dtype=np.float32) / 2.0
    tensor = torch.from_numpy(wafer_map).unsqueeze(0).unsqueeze(0)
    return F.interpolate(tensor, size=(image_size, image_size), mode="nearest").squeeze(0).squeeze(0).numpy()


def infer_label_from_row(row: pd.Series) -> str | None:
    failure = unwrap_legacy_value(row.get("failureType", "")).lower()
    if failure == "none":
        return LABEL_NORMAL
    if failure:
        return LABEL_DEFECT
    return None


class ArrayDataset(Dataset):
    def __init__(self, arrays: np.ndarray, labels: np.ndarray) -> None:
        self.arrays = arrays.astype(np.float32)
        self.labels = labels.astype(np.int64)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        x = torch.from_numpy(self.arrays[index]).unsqueeze(0)
        y = torch.tensor(int(self.labels[index]), dtype=torch.long)
        return x, y


def build_split_from_raw(config: dict[str, Any]) -> dict[str, np.ndarray]:
    raw_path = Path(config["run"]["raw_pickle"])
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw pickle not found: {raw_path}")

    image_size = int(config["split"]["image_size"])
    train_n = int(config["split"]["train_normal_count"])
    val_n = int(config["split"]["val_normal_count"])
    test_n = int(config["split"]["test_normal_count"])
    defect_fraction = float(config["split"]["test_defect_fraction_of_test_normals"])
    requested_defects = max(1, int(round(test_n * defect_fraction)))

    df = read_legacy_pickle(raw_path).copy()
    df["failureTypeText"] = df["failureType"].map(unwrap_legacy_value)
    df["label"] = df.apply(infer_label_from_row, axis=1)
    df = df[df["label"].notna()].reset_index(drop=True)
    normal_df = df[df["label"] == LABEL_NORMAL].sample(frac=1.0, random_state=config["run"]["seed"]).reset_index(drop=True)
    defect_df = df[df["label"] == LABEL_DEFECT].sample(frac=1.0, random_state=config["run"]["seed"]).reset_index(drop=True)

    required_normals = train_n + val_n + test_n
    if len(normal_df) < required_normals:
        raise ValueError(f"Need {required_normals} normals but found {len(normal_df)}")
    if len(defect_df) < requested_defects:
        raise ValueError(f"Need {requested_defects} defects but found {len(defect_df)}")

    train_df = normal_df.iloc[:train_n].copy()
    val_df = normal_df.iloc[train_n : train_n + val_n].copy()
    test_normal_df = normal_df.iloc[train_n + val_n : train_n + val_n + test_n].copy()
    test_defect_df = defect_df.iloc[:requested_defects].copy()
    test_df = pd.concat([test_normal_df, test_defect_df], ignore_index=True).sample(frac=1.0, random_state=config["run"]["seed"]).reset_index(drop=True)
    print(f"Prepared split | train normals={len(train_df)} | val normals={len(val_df)} | test normals={len(test_normal_df)} | test defects={len(test_defect_df)}")

    def to_arrays(frame: pd.DataFrame, split_name: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        arrays = np.stack([
            normalize_map(np.asarray(row["waferMap"]), image_size=image_size)
            for _, row in tqdm(frame.iterrows(), total=len(frame), desc=f"prepare_{split_name}", leave=True)
        ])
        labels = (frame["label"].values == LABEL_DEFECT).astype(np.int64)
        defect_types = frame["failureTypeText"].fillna("unlabeled").replace("", "unlabeled").to_numpy()
        return arrays, labels, defect_types

    train_x, train_y, train_defect_types = to_arrays(train_df, "train")
    val_x, val_y, val_defect_types = to_arrays(val_df, "val")
    test_x, test_y, test_defect_types = to_arrays(test_df, "test")
    return {
        "train_x": train_x,
        "train_y": train_y,
        "train_defect_types": train_defect_types,
        "val_x": val_x,
        "val_y": val_y,
        "val_defect_types": val_defect_types,
        "test_x": test_x,
        "test_y": test_y,
        "test_defect_types": test_defect_types,
    }


In [3]:
class ImageNetWaferPreprocessor(nn.Module):
    def __init__(self, input_size: int) -> None:
        super().__init__()
        self.input_size = input_size
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1, 3, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.repeat(1, 3, 1, 1)
        x = F.interpolate(x, size=(self.input_size, self.input_size), mode="bilinear", align_corners=False)
        return (x - self.mean) / self.std


class FrozenWideResNet50Backbone(nn.Module):
    def __init__(self, layers: list[str], input_size: int) -> None:
        super().__init__()
        backbone = wide_resnet50_2(weights=Wide_ResNet50_2_Weights.DEFAULT)
        self.preprocess = ImageNetWaferPreprocessor(input_size)
        self.extractor = create_feature_extractor(backbone, return_nodes={layer: layer for layer in layers})
        self.extractor.eval()
        for parameter in self.extractor.parameters():
            parameter.requires_grad_(False)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        x = self.preprocess(x)
        with torch.no_grad():
            return self.extractor(x)


class Orthogonal1x1Conv(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.weight_raw = nn.Parameter(torch.eye(channels, dtype=torch.float32))

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        q, _ = torch.linalg.qr(self.weight_raw)
        return F.conv2d(x, q.view(q.shape[0], q.shape[1], 1, 1)), torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)


class AffineCoupling(nn.Module):
    def __init__(self, channels: int, hidden_channels: int, scale_clamp: float) -> None:
        super().__init__()
        self.scale_clamp = scale_clamp
        half = channels // 2
        self.net = nn.Sequential(
            nn.Conv2d(half, hidden_channels, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(hidden_channels, hidden_channels, 1),
            nn.GELU(),
            nn.Conv2d(hidden_channels, channels, 3, padding=1),
        )
        final_layer = self.net[-1]
        nn.init.zeros_(final_layer.weight)
        nn.init.zeros_(final_layer.bias)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x_a, x_b = torch.chunk(x, 2, dim=1)
        shift, log_scale = torch.chunk(self.net(x_a), 2, dim=1)
        log_scale = torch.tanh(log_scale) * self.scale_clamp
        y_b = x_b * torch.exp(log_scale) + shift
        return torch.cat([x_a, y_b], dim=1), log_scale.flatten(1).sum(dim=1)


class FastFlowStage(nn.Module):
    def __init__(self, channels: int, flow_steps: int, hidden_channels: int, scale_clamp: float) -> None:
        super().__init__()
        self.mixers = nn.ModuleList([Orthogonal1x1Conv(channels) for _ in range(flow_steps)])
        self.couplings = nn.ModuleList([AffineCoupling(channels, hidden_channels, scale_clamp) for _ in range(flow_steps)])

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        total_logdet = torch.zeros(x.shape[0], device=x.device, dtype=x.dtype)
        z = x
        for mix, coupling in zip(self.mixers, self.couplings):
            z, logdet = mix(z)
            z, coupling_logdet = coupling(z)
            total_logdet = total_logdet + logdet + coupling_logdet
        return z, total_logdet


class FastFlowModel(nn.Module):
    def __init__(self, variant: dict[str, Any], image_size: int) -> None:
        super().__init__()
        self.backbone = FrozenWideResNet50Backbone(variant["layers"], variant["input_size"])
        with torch.no_grad():
            dummy = torch.zeros(1, 1, image_size, image_size)
            feature_shapes = self.backbone(dummy)
        self.stages = nn.ModuleDict({
            name: FastFlowStage(feature.shape[1], variant["flow_steps"], variant["hidden_channels"], variant["scale_clamp"])
            for name, feature in feature_shapes.items()
        })

    def forward(self, x: torch.Tensor) -> tuple[list[torch.Tensor], list[torch.Tensor]]:
        features = self.backbone(x)
        zs, logdets = [], []
        for name, feature in features.items():
            z, logdet = self.stages[name](feature)
            zs.append(z)
            logdets.append(logdet)
        return zs, logdets

    def anomaly_map(self, x: torch.Tensor, output_size: int) -> torch.Tensor:
        zs, _ = self.forward(x)
        maps = [F.interpolate(0.5 * torch.nan_to_num(z, nan=0.0, posinf=1e6, neginf=-1e6).pow(2).mean(dim=1, keepdim=True), size=(output_size, output_size), mode="bilinear", align_corners=False) for z in zs]
        return torch.nan_to_num(torch.stack(maps).mean(dim=0), nan=0.0, posinf=1e6, neginf=0.0)


def compute_loss(zs: list[torch.Tensor], logdets: list[torch.Tensor]) -> torch.Tensor:
    losses = []
    for z, logdet in zip(zs, logdets):
        nll = 0.5 * (z.pow(2) + LOG_2PI).flatten(1).sum(dim=1) - logdet
        losses.append((nll / z[0].numel()).mean())
    return torch.stack(losses).mean()


def reduce_scores(maps: torch.Tensor, reduction: str, topk_ratio: float) -> np.ndarray:
    maps = torch.nan_to_num(maps.detach().float(), nan=0.0, posinf=1e6, neginf=0.0)
    flat = maps.flatten(1)
    if reduction == "mean":
        scores = flat.mean(dim=1).numpy()
        return np.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=0.0)
    if reduction == "max":
        scores = flat.max(dim=1).values.numpy()
        return np.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=0.0)
    k = max(1, int(math.ceil(flat.shape[1] * topk_ratio)))
    scores = flat.topk(k=k, dim=1).values.mean(dim=1).numpy()
    return np.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=0.0)


def summarize_threshold_metrics(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict[str, Any]:
    non_finite_count = int((~np.isfinite(scores)).sum())
    if non_finite_count:
        print(f"Warning: replacing {non_finite_count} non-finite scores before metric computation")
    scores = np.nan_to_num(scores, nan=0.0, posinf=1e6, neginf=0.0)
    predicted = (scores > threshold).astype(int)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    balanced_accuracy = 0.5 * ((tp / max(1, tp + fn)) + (tn / max(1, tn + fp)))
    return {
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auroc": float(roc_auc_score(labels, scores)),
        "auprc": float(average_precision_score(labels, scores)),
        "balanced_accuracy": float(balanced_accuracy),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


def make_loaders(dataset: dict[str, np.ndarray], config: dict[str, Any], device: torch.device) -> dict[str, DataLoader]:
    bs = int(config["training"]["batch_size"])
    nw = int(config["training"]["num_workers"])
    pin = device.type == "cuda"
    return {
        "train": DataLoader(ArrayDataset(dataset["train_x"], dataset["train_y"]), batch_size=bs, shuffle=True, num_workers=nw, pin_memory=pin),
        "val": DataLoader(ArrayDataset(dataset["val_x"], dataset["val_y"]), batch_size=bs, shuffle=False, num_workers=nw, pin_memory=pin),
        "test": DataLoader(ArrayDataset(dataset["test_x"], dataset["test_y"]), batch_size=bs, shuffle=False, num_workers=nw, pin_memory=pin),
    }


def run_epoch(model: nn.Module, loader: DataLoader, device: torch.device, optimizer: torch.optim.Optimizer | None, scaler: torch.amp.GradScaler, amp_enabled: bool, grad_clip_norm: float | None, desc: str) -> float:
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss = 0.0
    total_items = 0
    progress = tqdm(loader, desc=desc, leave=False)
    for inputs, _ in progress:
        inputs = inputs.to(device, non_blocking=True)
        with torch.set_grad_enabled(train_mode):
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                zs, logdets = model(inputs)
                loss = compute_loss(zs, logdets)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss detected in {desc}. Try lowering learning_rate or batch_size.")
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                if grad_clip_norm is not None:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
        total_loss += float(loss.detach().cpu()) * inputs.shape[0]
        total_items += inputs.shape[0]
        progress.set_postfix(loss=f"{(total_loss / max(1, total_items)):.4f}")
    return total_loss / max(1, total_items)


def collect_maps(model: FastFlowModel, loader: DataLoader, device: torch.device, image_size: int, desc: str) -> tuple[torch.Tensor, np.ndarray]:
    model.eval()
    maps, labels = [], []
    with torch.no_grad():
        for inputs, batch_labels in tqdm(loader, desc=desc, leave=False):
            inputs = inputs.to(device, non_blocking=True)
            maps.append(model.anomaly_map(inputs, image_size).cpu())
            labels.append(batch_labels.numpy())
    return torch.cat(maps, dim=0), np.concatenate(labels, axis=0)


def build_defect_breakdown(defect_types: np.ndarray, labels: np.ndarray, scores: np.ndarray, threshold: float) -> pd.DataFrame:
    eval_df = pd.DataFrame({
        "defect_type": defect_types,
        "is_anomaly": labels.astype(int),
        "score": scores,
    })
    eval_df["predicted_anomaly"] = (eval_df["score"] > threshold).astype(int)
    defect_only = eval_df[eval_df["is_anomaly"] == 1].copy()
    grouped = defect_only.groupby("defect_type").agg(
        count=("defect_type", "size"),
        detected=("predicted_anomaly", "sum"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
    )
    grouped["missed"] = grouped["count"] - grouped["detected"]
    grouped["recall"] = grouped["detected"] / grouped["count"].clip(lower=1)
    return grouped.sort_values(["recall", "count", "mean_score"], ascending=[True, False, True]).reset_index()


def run_variant(dataset: dict[str, np.ndarray], variant: dict[str, Any], config: dict[str, Any]) -> dict[str, Any]:
    device = resolve_device(config["training"]["device"])
    loaders = make_loaders(dataset, config, device)
    model = FastFlowModel(variant, image_size=int(config["split"]["image_size"])) .to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["training"]["learning_rate"], weight_decay=config["training"]["weight_decay"])
    scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda" and config["training"]["amp"])
    best_state = None
    best_val_loss = float("inf")
    history = []
    stale = 0
    epoch_progress = tqdm(range(1, int(config["training"]["epochs"]) + 1), desc=f"epochs:{variant['name']}", leave=True)
    for epoch in epoch_progress:
        train_loss = run_epoch(model, loaders["train"], device, optimizer, scaler, device.type == "cuda" and config["training"]["amp"], config["training"]["grad_clip_norm"], desc=f"train:{variant['name']}:e{epoch}")
        val_loss = run_epoch(model, loaders["val"], device, None, scaler, False, None, desc=f"val:{variant['name']}:e{epoch}")
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        epoch_progress.set_postfix(train_loss=f"{train_loss:.4f}", val_loss=f"{val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= int(config["training"]["early_stopping_patience"]):
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    val_maps, _ = collect_maps(model, loaders["val"], device, int(config["split"]["image_size"]), desc=f"maps_val:{variant['name']}")
    test_maps, test_labels = collect_maps(model, loaders["test"], device, int(config["split"]["image_size"]), desc=f"maps_test:{variant['name']}")
    rows = []
    score_progress = tqdm(config["scoring"]["sweep_variants"], desc=f"scores:{variant['name']}", leave=False)
    for score_variant in score_progress:
        val_scores = reduce_scores(val_maps, score_variant["reduction"], score_variant["topk_ratio"])
        test_scores = reduce_scores(test_maps, score_variant["reduction"], score_variant["topk_ratio"])
        threshold = float(np.quantile(val_scores, config["scoring"]["threshold_quantile"]))
        metrics = summarize_threshold_metrics(test_labels.astype(int), test_scores, threshold)
        rows.append({"variant": variant["name"], "score_name": score_variant["name"], "reduction": score_variant["reduction"], "topk_ratio": score_variant["topk_ratio"], **metrics})
        score_progress.set_postfix(f1=f"{metrics['f1']:.4f}", recall=f"{metrics['recall']:.4f}")
    score_df = pd.DataFrame(rows).sort_values(["f1", "recall", "precision"], ascending=False).reset_index(drop=True)
    best_row = score_df.iloc[0].to_dict()
    best_threshold = float(best_row["threshold"])
    best_scores = reduce_scores(test_maps, best_row["reduction"], float(best_row["topk_ratio"]))
    defect_breakdown_df = build_defect_breakdown(dataset["test_defect_types"], test_labels.astype(int), best_scores, best_threshold)
    hard_defects = defect_breakdown_df.head(8)
    print(f"Hardest defect types for {variant['name']}:")
    display(hard_defects)
    return {"history": pd.DataFrame(history), "score_df": score_df, "best": best_row, "defect_breakdown_df": defect_breakdown_df}


def prepare_output_dir(config: dict[str, Any]) -> Path:
    output_dir = Path(config["run"]["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir


def dataset_cache_path(config: dict[str, Any]) -> Path:
    return prepare_output_dir(config) / "dataset_cache.npz"


def save_dataset_cache(dataset: dict[str, np.ndarray], cache_path: Path) -> None:
    np.savez_compressed(cache_path, **dataset)
    print(f"Saved dataset cache to {cache_path}")


def load_dataset_cache(cache_path: Path) -> dict[str, np.ndarray]:
    cached = np.load(cache_path, allow_pickle=True)
    dataset = {key: cached[key] for key in cached.files}
    print(f"Loaded dataset cache from {cache_path}")
    return dataset


def prepare_dataset(config: dict[str, Any], rebuild: bool = False) -> dict[str, np.ndarray]:
    set_seed(config["run"]["seed"])
    cache_path = dataset_cache_path(config)
    if cache_path.exists() and not rebuild:
        return load_dataset_cache(cache_path)
    dataset = build_split_from_raw(config)
    save_dataset_cache(dataset, cache_path)
    return dataset


def get_variant(config: dict[str, Any], variant_name: str) -> dict[str, Any]:
    for variant in config["variants"]:
        if variant["name"] == variant_name:
            return variant
    raise KeyError(f"Unknown variant: {variant_name}")


def save_variant_result(output_dir: Path, variant_name: str, result: dict[str, Any]) -> None:
    result["history"].to_csv(output_dir / f"{variant_name}_history.csv", index=False)
    result["score_df"].to_csv(output_dir / f"{variant_name}_scores.csv", index=False)
    result["defect_breakdown_df"].to_csv(output_dir / f"{variant_name}_defect_breakdown.csv", index=False)
    pd.DataFrame([result["best"]]).to_csv(output_dir / f"{variant_name}_best_row.csv", index=False)
    print(f"Saved outputs for {variant_name} to {output_dir}")


def run_and_save_variant(dataset: dict[str, np.ndarray], config: dict[str, Any], variant_name: str) -> dict[str, Any]:
    output_dir = prepare_output_dir(config)
    variant = get_variant(config, variant_name)
    print(f"Running {variant_name}")
    result = run_variant(dataset, variant, config)
    save_variant_result(output_dir, variant_name, result)
    return result


def load_saved_summary(config: dict[str, Any]) -> pd.DataFrame:
    output_dir = prepare_output_dir(config)
    best_rows = []
    for variant in config["variants"]:
        best_path = output_dir / f"{variant['name']}_best_row.csv"
        if best_path.exists():
            best_rows.append(pd.read_csv(best_path).iloc[0].to_dict())
    if not best_rows:
        raise FileNotFoundError("No saved variant results found yet.")
    summary_df = pd.DataFrame(best_rows).sort_values(["f1", "recall", "precision"], ascending=False).reset_index(drop=True)
    summary_df.to_csv(output_dir / "fastflow_variant_summary.csv", index=False)
    return summary_df


In [4]:
cache_path = dataset_cache_path(CONFIG)
if cache_path.exists():
    cache_path.unlink()
    print(f"Deleted corrupted cache: {cache_path}")


In [5]:
output_dir = prepare_output_dir(CONFIG)
dataset = prepare_dataset(CONFIG, rebuild=True)
print(f"Output dir: {output_dir}")
print({key: value.shape if hasattr(value, 'shape') else type(value) for key, value in dataset.items()})


Prepared split | train normals=50000 | val normals=5000 | test normals=5000 | test defects=250


prepare_train:   0%|          | 0/50000 [00:00<?, ?it/s]

prepare_val:   0%|          | 0/5000 [00:00<?, ?it/s]

prepare_test:   0%|          | 0/5250 [00:00<?, ?it/s]

Saved dataset cache to /output/fastflow_all_in_one/dataset_cache.npz
Output dir: /output/fastflow_all_in_one
{'train_x': (50000, 64, 64), 'train_y': (50000,), 'train_defect_types': (50000,), 'val_x': (5000, 64, 64), 'val_y': (5000,), 'val_defect_types': (5000,), 'test_x': (5250, 64, 64), 'test_y': (5250,), 'test_defect_types': (5250,)}


In [11]:
result_wrn50_l23_s6 = run_and_save_variant(dataset, CONFIG, "wrn50_l23_s6")
display(result_wrn50_l23_s6["score_df"])
display(result_wrn50_l23_s6["defect_breakdown_df"].head(12))


Running wrn50_l23_s6


epochs:wrn50_l23_s6:   0%|          | 0/16 [00:00<?, ?it/s]

train:wrn50_l23_s6:e1:   0%|          | 0/391 [00:00<?, ?it/s]

val:wrn50_l23_s6:e1:   0%|          | 0/40 [00:00<?, ?it/s]

train:wrn50_l23_s6:e2:   0%|          | 0/391 [00:00<?, ?it/s]

...

Hardest defect types for wrn50_l23_s6:


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Scratch,15,2,0.419726,0.407645,13,0.133333
1,Edge-Loc,53,27,1816.209961,0.438789,26,0.509434
2,Loc,34,18,3780.458008,0.443858,16,0.529412
3,Donut,7,4,9430.153320,0.485883,3,0.571429
4,Center,50,34,4820.661621,1.052170,16,0.680000
5,Edge-Ring,84,65,0.458891,0.453491,19,0.773810
6,Random,5,5,158.765778,0.547538,0,1.000000
7,Near-full,2,2,59081.656250,59081.655273,0,1.000000


Saved outputs for wrn50_l23_s6 to /output/fastflow_all_in_one


,variant,score_name,reduction,topk_ratio,threshold,precision,recall,f1,auroc,auprc,balanced_accuracy,tp,fp,tn,fn
0,wrn50_l23_s6,mean,mean,0.10,0.437844,0.398477,0.628,0.487578,0.876862,0.513479,0.7903,157,237,4763,93
1,wrn50_l23_s6,topk_r015,topk_mean,0.15,0.616327,0.413793,0.576,0.481605,0.894372,0.480494,0.7676,144,204,4796,106
2,wrn50_l23_s6,topk_r010,topk_mean,0.10,0.655969,0.370787,0.528,0.435644,0.887818,0.456219,0.7416,132,224,4776,118
3,wrn50_l23_s6,topk_r005,topk_mean,0.05,0.750224,0.327434,0.444,0.376910,0.872825,0.420182,0.6992,111,228,4772,139
4,wrn50_l23_s6,max,max,0.10,1.600702,0.261538,0.340,0.295652,0.796934,0.356195,0.6460,85,240,4760,165


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Scratch,15,2,0.419726,0.407645,13,0.133333
1,Edge-Loc,53,27,1816.209961,0.438789,26,0.509434
2,Loc,34,18,3780.458008,0.443858,16,0.529412
3,Donut,7,4,9430.153320,0.485883,3,0.571429
4,Center,50,34,4820.661621,1.052170,16,0.680000
5,Edge-Ring,84,65,0.458891,0.453491,19,0.773810
6,Random,5,5,158.765778,0.547538,0,1.000000
7,Near-full,2,2,59081.656250,59081.655273,0,1.000000


In [6]:
result_wrn50_l2_s6 = run_and_save_variant(dataset, CONFIG, "wrn50_l2_s6")
display(result_wrn50_l2_s6["score_df"])
display(result_wrn50_l2_s6["defect_breakdown_df"].head(12))


Running wrn50_l2_s6
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-9ba9bcbe.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-9ba9bcbe.pth


100%|████████████████████████████████████████████████████████████████████████| 263M/263M [00:02<00:00, 108MB/s]


epochs:wrn50_l2_s6:   0%|          | 0/16 [00:00<?, ?it/s]

train:wrn50_l2_s6:e1:   0%|          | 0/391 [00:00<?, ?it/s]

val:wrn50_l2_s6:e1:   0%|          | 0/40 [00:00<?, ?it/s]

...

Hardest defect types for wrn50_l2_s6:


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Scratch,15,1,0.645252,0.626173,14,0.066667
1,Edge-Loc,53,21,1462.942627,0.703745,32,0.396226
2,Donut,7,4,7754.627441,0.873769,3,0.571429
3,Loc,34,20,2343.144775,0.749196,14,0.588235
4,Center,50,32,3515.175049,3.620339,18,0.640000
5,Edge-Ring,84,57,0.774717,0.745125,27,0.678571
6,Random,5,5,18.891796,1.078065,0,1.000000
7,Near-full,2,2,37013.769531,37013.768555,0,1.000000


Saved outputs for wrn50_l2_s6 to /output/fastflow_all_in_one


,variant,score_name,reduction,topk_ratio,threshold,precision,recall,f1,auroc,auprc,balanced_accuracy,tp,fp,tn,fn
0,wrn50_l2_s6,topk_r015,topk_mean,0.15,0.723919,0.405714,0.568,0.473333,0.891069,0.492186,0.7632,142,208,4792,108
1,wrn50_l2_s6,mean,mean,0.10,0.461831,0.388740,0.580,0.465490,0.854968,0.491881,0.7672,145,228,4772,105
2,wrn50_l2_s6,topk_r010,topk_mean,0.10,0.785838,0.389881,0.524,0.447099,0.885685,0.469326,0.7415,131,205,4795,119
3,wrn50_l2_s6,topk_r005,topk_mean,0.05,0.925677,0.332344,0.448,0.381601,0.871851,0.428562,0.7015,112,225,4775,138
4,wrn50_l2_s6,max,max,0.10,2.570242,0.275316,0.348,0.307420,0.787256,0.363108,0.6511,87,229,4771,163


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Scratch,15,1,0.645252,0.626173,14,0.066667
1,Edge-Loc,53,21,1462.942627,0.703745,32,0.396226
2,Donut,7,4,7754.627441,0.873769,3,0.571429
3,Loc,34,20,2343.144775,0.749196,14,0.588235
4,Center,50,32,3515.175049,3.620339,18,0.640000
5,Edge-Ring,84,57,0.774717,0.745125,27,0.678571
6,Random,5,5,18.891796,1.078065,0,1.000000
7,Near-full,2,2,37013.769531,37013.768555,0,1.000000


In [7]:
result_wrn50_l23_s4 = run_and_save_variant(dataset, CONFIG, "wrn50_l23_s4")
display(result_wrn50_l23_s4["score_df"])
display(result_wrn50_l23_s4["defect_breakdown_df"].head(12))


Running wrn50_l23_s4


epochs:wrn50_l23_s4:   0%|          | 0/16 [00:00<?, ?it/s]

train:wrn50_l23_s4:e1:   0%|          | 0/391 [00:00<?, ?it/s]

val:wrn50_l23_s4:e1:   0%|          | 0/40 [00:00<?, ?it/s]

train:wrn50_l23_s4:e2:   0%|          | 0/391 [00:00<?, ?it/s]

...

Hardest defect types for wrn50_l23_s4:


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Scratch,15,3,0.377202,0.363949,12,0.200000
1,Edge-Loc,53,29,148.378906,0.401662,24,0.547170
2,Loc,34,19,324.178894,0.401371,15,0.558824
3,Donut,7,5,734.070435,0.454123,2,0.714286
4,Center,50,36,560.050659,0.934789,14,0.720000
5,Edge-Ring,84,67,0.414847,0.414894,17,0.797619
6,Random,5,5,8.352972,0.520185,0,1.000000
7,Near-full,2,2,4615.229004,4615.229004,0,1.000000


Saved outputs for wrn50_l23_s4 to /output/fastflow_all_in_one


,variant,score_name,reduction,topk_ratio,threshold,precision,recall,f1,auroc,auprc,balanced_accuracy,tp,fp,tn,fn
0,wrn50_l23_s4,mean,mean,0.10,0.396222,0.420253,0.664,0.514729,0.880458,0.533389,0.8091,166,229,4771,84
1,wrn50_l23_s4,topk_r015,topk_mean,0.15,0.591237,0.426593,0.616,0.504092,0.900715,0.503875,0.7873,154,207,4793,96
2,wrn50_l23_s4,topk_r010,topk_mean,0.10,0.630739,0.399449,0.580,0.473083,0.894947,0.481436,0.7682,145,218,4782,105
3,wrn50_l23_s4,topk_r005,topk_mean,0.05,0.712093,0.355114,0.500,0.415282,0.881003,0.440664,0.7273,125,227,4773,125
4,wrn50_l23_s4,max,max,0.10,1.505114,0.271028,0.348,0.304729,0.802977,0.364477,0.6506,87,234,4766,163


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Scratch,15,3,0.377202,0.363949,12,0.200000
1,Edge-Loc,53,29,148.378906,0.401662,24,0.547170
2,Loc,34,19,324.178894,0.401371,15,0.558824
3,Donut,7,5,734.070435,0.454123,2,0.714286
4,Center,50,36,560.050659,0.934789,14,0.720000
5,Edge-Ring,84,67,0.414847,0.414894,17,0.797619
6,Random,5,5,8.352972,0.520185,0,1.000000
7,Near-full,2,2,4615.229004,4615.229004,0,1.000000


In [8]:
summary_df = load_saved_summary(CONFIG)
display(summary_df)
best_variant_name = str(summary_df.iloc[0]["variant"])
best_defect_breakdown_path = output_dir / f"{best_variant_name}_defect_breakdown.csv"
print(f"Best variant: {best_variant_name}")
print(f"Best defect-breakdown CSV: {best_defect_breakdown_path}")
display(pd.read_csv(best_defect_breakdown_path).head(12))


,variant,score_name,reduction,topk_ratio,threshold,precision,recall,f1,auroc,auprc,balanced_accuracy,tp,fp,tn,fn
0,wrn50_l23_s4,mean,mean,0.10,0.396222,0.420253,0.664,0.514729,0.880458,0.533389,0.8091,166,229,4771,84
1,wrn50_l2_s6,topk_r015,topk_mean,0.15,0.723919,0.405714,0.568,0.473333,0.891069,0.492186,0.7632,142,208,4792,108


Best variant: wrn50_l23_s4
Best defect-breakdown CSV: /output/fastflow_all_in_one/wrn50_l23_s4_defect_breakdown.csv


,defect_type,count,detected,mean_score,median_score,missed,recall
0,Scratch,15,3,0.377202,0.363949,12,0.200000
1,Edge-Loc,53,29,148.378900,0.401662,24,0.547170
2,Loc,34,19,324.178900,0.401371,15,0.558824
3,Donut,7,5,734.070430,0.454123,2,0.714286
4,Center,50,36,560.050660,0.934789,14,0.720000
5,Edge-Ring,84,67,0.414847,0.414894,17,0.797619
6,Random,5,5,8.352972,0.520185,0,1.000000
7,Near-full,2,2,4615.229000,4615.229000,0,1.000000
